# Compare built-in OMPs with JLM and JLMB

This notebook compares the built-in phenomenological optical models `kduq`,
`chuq`, and `wlh` with the folding-based `jlm` and `jlmb` central potentials
for the elastic reaction $n + ^{208}$Pb.

For the built-in models, the single reference curves are not all the same kind
of object:

- **KDUQ** uses the original **KD03 frequentist** parameter set.
- **CHUQ** uses the original **CH89 frequentist** parameter set.
- **WLH** uses the packaged **WLH mean** parameter set.

We propagate the packaged posterior samples for KDUQ, CHUQ, and WLH into both
the differential cross sections and the central volume integrals, while JLM and
JLMB remain deterministic folding baselines.


## Imports


In [ ]:
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import jitr
from jitr.folding import TwoParameterFermiDensity
from jitr.folding.jlm import (
    JLMParameters,
    JLMPotential,
    JLMSelfEnergy,
    make_jlmb_parameters,
)
from jitr.optical_potentials import chuq, kduq, wlh
from jitr.reactions import ElasticReaction
from jitr.rmatrix import Solver
from jitr.xs.elastic import DifferentialWorkspace


## Reaction setup

We keep the same benchmark system as before: neutron elastic scattering on
$^{208}$Pb at 14 MeV. To keep the notebook quick enough for nbval, we draw a
fixed-size reproducible subset from each packaged posterior instead of running
every available sample.


In [ ]:
target = (208, 82)
projectile = (1, 0)  # neutron
energy_lab = 14.0
angles_deg = np.linspace(5.0, 175.0, 121)
angles_rad = np.deg2rad(angles_deg)
energy_grid = np.linspace(5.0, 40.0, 8)

max_posterior_draws = 128
posterior_seed = 208

reaction = ElasticReaction(target=target, projectile=projectile)
kinematics = reaction.kinematics(energy_lab)
a = jitr.utils.interaction_range(target[0]) * kinematics.k + np.pi * 3
N = jitr.utils.suggested_basis_size(a)
channel_radius_fm = a / kinematics.k
workspace = DifferentialWorkspace.build_from_system(
    reaction=reaction,
    kinematics=kinematics,
    channel_radius_fm=channel_radius_fm,
    solver=Solver(N),
    lmax=50,
    angles=angles_rad,
)
rgrid = workspace.radial_grid()
volume_r = np.linspace(0.0, 15.0, 1200)
A = target[0]
rng = np.random.default_rng(posterior_seed)


## Shared helpers


In [ ]:
rho_n_pb = TwoParameterFermiDensity(R=6.7, a=0.55, N=126)
rho_p_pb = TwoParameterFermiDensity(R=6.654, a=0.547, N=82)


@dataclass
class BaselineResult:
    label: str
    reference_label: str
    xs: object
    JV: float
    JW: float


@dataclass
class PosteriorResult:
    label: str
    source: str
    ndraws: int
    xs_band: np.ndarray
    sigma_tot_band: np.ndarray
    sigma_rxn_band: np.ndarray
    JV_band: np.ndarray
    JW_band: np.ndarray
    JV_energy_band: np.ndarray
    JW_energy_band: np.ndarray


def wlh_mean_parameters(projectile):
    return np.asarray(list(wlh.Global(projectile).params.values()), dtype=np.float64)


builtin_models = {
    'KDUQ': {
        'module': kduq,
        'global': kduq.Global(projectile),
        'reference_sample': kduq.get_kd03(projectile),
        'reference_label': 'KD03 frequentist parameter set',
        'all_samples': kduq.get_samples(projectile, posterior='federal'),
        'posterior_label': 'Federal posterior',
    },
    'CHUQ': {
        'module': chuq,
        'global': chuq.Global(),
        'reference_sample': chuq.get_ch89(),
        'reference_label': 'CH89 frequentist parameter set',
        'all_samples': chuq.get_samples(posterior='federal'),
        'posterior_label': 'Federal posterior',
    },
    'WLH': {
        'module': wlh,
        'global': wlh.Global(projectile),
        'reference_sample': wlh_mean_parameters(projectile),
        'reference_label': 'WLH mean parameter set',
        'all_samples': wlh.get_samples(projectile),
        'posterior_label': 'Packaged posterior ensemble',
    },
}

colors = {
    'KDUQ': 'tab:blue',
    'CHUQ': 'tab:orange',
    'WLH': 'tab:green',
    'JLM': 'tab:red',
    'JLMB': 'tab:purple',
}


def safe_radius_grid(r):
    radius = np.asarray(r, dtype=float)
    return np.where(radius == 0.0, np.finfo(float).eps, radius)


def central_volume_integrals(central_term, r):
    jv = -4.0 * np.pi * np.trapezoid(np.real(central_term) * r**2, r) / A
    jw = -4.0 * np.pi * np.trapezoid(np.imag(central_term) * r**2, r) / A
    return float(jv), float(jw)


def percentile_band(samples):
    return np.percentile(samples, [16.0, 50.0, 84.0], axis=0)


def format_band(band, digits=1):
    return f"{band[1]:.{digits}f} [{band[0]:.{digits}f}, {band[2]:.{digits}f}]"


def select_posterior_draws(all_samples, max_draws):
    all_samples = np.asarray(all_samples)
    if max_draws >= len(all_samples):
        return all_samples
    selection = np.sort(rng.choice(len(all_samples), size=max_draws, replace=False))
    return all_samples[selection]


def builtin_params(module, energy, sample):
    return module.calculate_params(projectile, target, energy, *sample)


def builtin_terms(module, energy, r, sample):
    central_params, spin_orbit_params, _ = builtin_params(module, energy, sample)
    safe_r = safe_radius_grid(r)
    central_term = module.central(safe_r, *central_params)
    spin_orbit_term = module.spin_orbit(safe_r, *spin_orbit_params)
    return central_term, spin_orbit_term


def make_jlm_potential(energy, use_jlmb=False):
    params = make_jlmb_parameters(energy) if use_jlmb else JLMParameters()
    return JLMPotential(rho_n_pb, rho_p_pb, JLMSelfEnergy('n'), parameters=params)


def jlm_terms(energy, use_jlmb=False, r=None):
    potential = make_jlm_potential(energy, use_jlmb=use_jlmb)
    real_term, imag_term = potential.compute(r, E=energy, projectile='n')
    central_term = np.asarray(real_term + 1j * imag_term, dtype=np.complex128)
    return potential, central_term


def summarize_builtin_posterior(label, module, samples):
    xs_samples = np.zeros((len(samples), angles_deg.size))
    sigma_tot_samples = np.zeros(len(samples))
    sigma_rxn_samples = np.zeros(len(samples))
    JV_samples = np.zeros(len(samples))
    JW_samples = np.zeros(len(samples))
    JV_energy_samples = np.zeros((len(samples), energy_grid.size))
    JW_energy_samples = np.zeros((len(samples), energy_grid.size))

    for i, sample in enumerate(samples):
        central_term, spin_orbit_term = builtin_terms(module, energy_lab, rgrid, sample)
        xs = workspace.xs(central_term, spin_orbit_term)
        xs_samples[i] = xs.dsdo
        sigma_tot_samples[i] = float(xs.t)
        sigma_rxn_samples[i] = float(xs.rxn)

        central_volume_term, _ = builtin_terms(module, energy_lab, volume_r, sample)
        JV_samples[i], JW_samples[i] = central_volume_integrals(central_volume_term, volume_r)

        for j, energy in enumerate(energy_grid):
            central_energy_term, _ = builtin_terms(module, energy, volume_r, sample)
            JV_energy_samples[i, j], JW_energy_samples[i, j] = central_volume_integrals(
                central_energy_term, volume_r
            )

    return PosteriorResult(
        label=label,
        source=builtin_models[label]['posterior_label'],
        ndraws=len(samples),
        xs_band=percentile_band(xs_samples),
        sigma_tot_band=percentile_band(sigma_tot_samples),
        sigma_rxn_band=percentile_band(sigma_rxn_samples),
        JV_band=percentile_band(JV_samples),
        JW_band=percentile_band(JW_samples),
        JV_energy_band=percentile_band(JV_energy_samples),
        JW_energy_band=percentile_band(JW_energy_samples),
    )


## Reference parameter sets and posterior draw counts


In [ ]:
posterior_samples = {
    label: select_posterior_draws(config['all_samples'], max_posterior_draws)
    for label, config in builtin_models.items()
}

pd.DataFrame(
    {
        'Model': list(posterior_samples.keys()),
        'Reference curve': [builtin_models[label]['reference_label'] for label in posterior_samples],
        'Posterior source': [builtin_models[label]['posterior_label'] for label in posterior_samples],
        'Available samples': [len(builtin_models[label]['all_samples']) for label in posterior_samples],
        'Draws used here': [len(samples) for samples in posterior_samples.values()],
    }
)


## Reference parameter-set evaluations at 14 MeV


In [ ]:
reference_results = {}

for label, config in builtin_models.items():
    central_term, spin_orbit_term = builtin_terms(
        config['module'], energy_lab, rgrid, config['reference_sample']
    )
    xs = workspace.xs(central_term, spin_orbit_term)
    central_volume_term, _ = builtin_terms(
        config['module'], energy_lab, volume_r, config['reference_sample']
    )
    JV, JW = central_volume_integrals(central_volume_term, volume_r)
    reference_results[label] = BaselineResult(
        label=label,
        reference_label=config['reference_label'],
        xs=xs,
        JV=JV,
        JW=JW,
    )

for label, use_jlmb in [('JLM', False), ('JLMB', True)]:
    potential, central_term = jlm_terms(energy_lab, use_jlmb=use_jlmb, r=rgrid)
    xs = workspace.xs(central_term)
    JV, JW = potential.volume_integrals(energy_lab, 'n', A)
    reference_results[label] = BaselineResult(
        label=label,
        reference_label='deterministic folding model',
        xs=xs,
        JV=JV,
        JW=JW,
    )


## Posterior summaries for KDUQ, CHUQ, and WLH


In [ ]:
posterior_results = {
    label: summarize_builtin_posterior(
        label,
        config['module'],
        posterior_samples[label],
    )
    for label, config in builtin_models.items()
}


## Differential cross sections at 14 MeV

The shaded regions show the 16th--84th percentile posterior intervals for KDUQ,
CHUQ, and WLH. The corresponding reference curves are KD03, CH89, and the WLH
mean parameter set. JLM and JLMB are plotted as deterministic folding baselines.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for label in ['KDUQ', 'CHUQ', 'WLH']:
    color = colors[label]
    reference_xs = reference_results[label].xs.dsdo
    band = posterior_results[label].xs_band
    ax.fill_between(
        angles_deg,
        band[0],
        band[2],
        color=color,
        alpha=0.18,
        label=f'{label} 68% posterior',
    )
    ax.plot(
        angles_deg,
        reference_xs,
        color=color,
        lw=2,
        label=f"{label} {reference_results[label].reference_label}",
    )

for label in ['JLM', 'JLMB']:
    ax.plot(
        angles_deg,
        reference_results[label].xs.dsdo,
        color=colors[label],
        lw=2,
        ls='--',
        label=f'{label} deterministic folding model',
    )

ax.set_yscale('log')
ax.set_xlabel('Angle (deg)')
ax.set_ylabel(r'$d\sigma/d\Omega$ (mb/sr)')
ax.set_title(r'$n + ^{208}$Pb at 14 MeV')
ax.grid(alpha=0.3)
ax.legend(ncol=2, fontsize=9)
plt.show()


## Benchmark summary table

The built-in OMP rows distinguish the single reference parameter set from the
posterior interval summary. JLM and JLMB remain deterministic folding models.


In [ ]:
summary_rows = []

for label in ['KDUQ', 'CHUQ', 'WLH']:
    ref = reference_results[label]
    post = posterior_results[label]
    summary_rows.append(
        {
            'Model': label,
            'Reference curve': ref.reference_label,
            'Posterior source': f"{post.source} ({post.ndraws} draws)",
            'J_V/A reference': round(ref.JV, 1),
            'J_V/A 68% CI': format_band(post.JV_band),
            'J_W/A reference': round(ref.JW, 1),
            'J_W/A 68% CI': format_band(post.JW_band),
            'sigma_tot reference (mb)': round(float(ref.xs.t), 1),
            'sigma_tot 68% CI (mb)': format_band(post.sigma_tot_band),
            'sigma_rxn reference (mb)': round(float(ref.xs.rxn), 1),
            'sigma_rxn 68% CI (mb)': format_band(post.sigma_rxn_band),
        }
    )

for label in ['JLM', 'JLMB']:
    ref = reference_results[label]
    summary_rows.append(
        {
            'Model': label,
            'Reference curve': ref.reference_label,
            'Posterior source': '--',
            'J_V/A reference': round(ref.JV, 1),
            'J_V/A 68% CI': '--',
            'J_W/A reference': round(ref.JW, 1),
            'J_W/A 68% CI': '--',
            'sigma_tot reference (mb)': round(float(ref.xs.t), 1),
            'sigma_tot 68% CI (mb)': '--',
            'sigma_rxn reference (mb)': round(float(ref.xs.rxn), 1),
            'sigma_rxn 68% CI (mb)': '--',
        }
    )

pd.DataFrame(summary_rows)


## Central volume integrals versus energy

For the built-in OMPs, the solid lines show the median posterior prediction and
the shaded regions show the 16th--84th percentile intervals. JLM and JLMB are
shown as deterministic folding curves.


In [ ]:
jlm_energy_curves = {name: {'JV': [], 'JW': []} for name in ['JLM', 'JLMB']}

for energy in energy_grid:
    for label, use_jlmb in [('JLM', False), ('JLMB', True)]:
        potential = make_jlm_potential(energy, use_jlmb=use_jlmb)
        JV, JW = potential.volume_integrals(energy, 'n', A)
        jlm_energy_curves[label]['JV'].append(JV)
        jlm_energy_curves[label]['JW'].append(JW)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)

for label in ['KDUQ', 'CHUQ', 'WLH']:
    color = colors[label]
    JV_band = posterior_results[label].JV_energy_band
    JW_band = posterior_results[label].JW_energy_band
    axes[0].fill_between(energy_grid, JV_band[0], JV_band[2], color=color, alpha=0.18)
    axes[0].plot(energy_grid, JV_band[1], color=color, lw=2, label=f'{label} posterior median')
    axes[1].fill_between(energy_grid, JW_band[0], JW_band[2], color=color, alpha=0.18)
    axes[1].plot(energy_grid, JW_band[1], color=color, lw=2, label=f'{label} posterior median')

for label in ['JLM', 'JLMB']:
    axes[0].plot(
        energy_grid,
        jlm_energy_curves[label]['JV'],
        color=colors[label],
        lw=2,
        ls='--',
        label=f'{label} deterministic',
    )
    axes[1].plot(
        energy_grid,
        jlm_energy_curves[label]['JW'],
        color=colors[label],
        lw=2,
        ls='--',
        label=f'{label} deterministic',
    )

axes[0].set_ylabel(r'$J_V/A$ (MeV fm$^3$)')
axes[1].set_ylabel(r'$J_W/A$ (MeV fm$^3$)')
for ax in axes:
    ax.set_xlabel('Laboratory energy (MeV)')
    ax.grid(alpha=0.3)
axes[0].set_title('Real central volume integral')
axes[1].set_title('Imaginary central volume integral')
axes[1].legend(loc='best')
fig.tight_layout()
plt.show()
